# Chapter 17: Attribution

In [1]:
import numpy as np
from expkit.attribution.touch import first_touch, last_touch, linear, time_decay, aggregate_credit
from expkit.plot.style import apply_style
apply_style()

In [2]:
touches = ['search', 'social', 'email', 'direct']
times = [0, 5, 10, 15]
for fn in [first_touch, last_touch, linear]:
    print(f'{fn.__name__:<14}', fn(touches))
print(f'{"time_decay":<14}', time_decay(touches, times, half_life=7))

first_touch    {'search': 1.0}
last_touch     {'direct': 1.0}
linear         {'search': 0.25, 'social': 0.25, 'email': 0.25, 'direct': 0.25}
time_decay     {'search': 0.1025764206563603, 'social': 0.1682941291142491, 'email': 0.2761152486418658, 'direct': 0.4530142015875249}


In [3]:
rng = np.random.default_rng(170)
channels = ['search', 'social', 'email', 'display', 'direct']
true_coef = {'search': 0.10, 'social': 0.04, 'email': 0.05, 'display': 0.02, 'direct': 0.20}
journeys = []
for _ in range(5000):
    nt = rng.integers(1, 6)
    t = list(rng.choice(channels, size=nt))
    times = sorted(rng.uniform(0, 30, size=nt))
    score = sum(true_coef[c] for c in t)
    converted = int(rng.random() < 1/(1+np.exp(-(score-0.5))))
    journeys.append((t, list(times), converted))
for scheme in ['first', 'last', 'linear', 'time_decay']:
    df = aggregate_credit(journeys, scheme=scheme)
    df['share'] = df['credit'] / df['credit'].sum()
    print(f'\n--- {scheme} ---'); print(df.to_string(index=False))


--- first ---
channel  credit    share
 direct   522.0 0.238683
display   420.0 0.192044
  email   422.0 0.192958
 search   433.0 0.197988
 social   390.0 0.178326

--- last ---
channel  credit    share
 direct   473.0 0.216278
display   431.0 0.197074
  email   403.0 0.184271
 search   442.0 0.202103
 social   438.0 0.200274

--- linear ---
channel     credit    share
 direct 478.216667 0.218663
display 428.466667 0.195915
  email 414.100000 0.189346
 search 443.983333 0.203010
 social 422.233333 0.193065

--- time_decay ---
channel     credit    share
 direct 465.368988 0.212789
display 430.317970 0.196762
  email 413.506329 0.189075
 search 450.460263 0.205972
 social 427.346450 0.195403
